# 📘 Feature Extraction & Descriptor

---

# 1. Import Library

```python
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
```

---

# 2. Load Contoh Dataset

```python
img = cv2.imread('image.jpg',0)
plt.imshow(img,cmap='gray')
plt.title('Input Image')
plt.axis('off')
```

---

# 🧩 REGION DESCRIPTOR

## Konsep

Mendeskripsikan bentuk objek berdasarkan area dan geometri.

Fitur umum:

* Area
* Perimeter
* Centroid
* Bounding box

```python
_,binary = cv2.threshold(img,127,255,cv2.THRESH_BINARY)
contours,_ = cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

cnt = contours[0]
area = cv2.contourArea(cnt)
perimeter = cv2.arcLength(cnt,True)
print(area, perimeter)
```

---

# 📐 MOMENT DESCRIPTOR

## Konsep

Moments menggambarkan distribusi massa piksel.

```python
moments = cv2.moments(binary)
cx = moments['m10']/moments['m00']
cy = moments['m01']/moments['m00']
print(cx,cy)
```

---

# 🔁 MOMENT INVARIANTS (HU MOMENTS)

Invariant terhadap rotasi, translasi, dan skala.

```python
hu = cv2.HuMoments(moments)
print(hu)
```

---

# 🌾 TEXTURE DESCRIPTOR

## Konsep

Texture = variasi lokal intensitas.

```python
mean = np.mean(img)
std = np.std(img)
energy = np.sum(img**2)
print(mean,std,energy)
```

---

# 🌈 SPECTRAL TEXTURE (FFT)

## Analisis domain frekuensi.

```python
f = np.fft.fft2(img)
fshift = np.fft.fftshift(f)
spectrum = np.log(np.abs(fshift)+1)

plt.imshow(spectrum,cmap='gray')
plt.title('Frequency Spectrum')
plt.axis('off')
```

---

# 🔲 CO-OCCURRENCE MATRIX (GLCM)

Mengukur hubungan piksel bertetangga.

```python
glcm = graycomatrix(img,
                    distances=[1],
                    angles=[0],
                    levels=256,
                    symmetric=True,
                    normed=True)

contrast = graycoprops(glcm,'contrast')[0,0]
homogeneity = graycoprops(glcm,'homogeneity')[0,0]
print(contrast, homogeneity)
```

---

# 🔥 LOCAL BINARY PATTERN (LBP)

Mendeskripsikan tekstur lokal.

```python
lbp = local_binary_pattern(img,P=8,R=1,method='uniform')

plt.imshow(lbp,cmap='gray')
plt.title('LBP Image')
plt.axis('off')
```

Histogram LBP sebagai feature vector:

```python
(hist,_)=np.histogram(lbp.ravel(),bins=10,range=(0,10))
hist = hist.astype('float')
hist /= hist.sum()
```

---

# 🧮 PRINCIPAL COMPONENT ANALYSIS (PCA)

Reduksi dimensi fitur.

```python
X = np.random.rand(100,50)

pca = PCA(n_components=10)
X_reduced = pca.fit_transform(X)
print(X_reduced.shape)
```

---

# 🤖 FEATURE VECTOR

Gabungkan beberapa descriptor.

```python
feature_vector = np.hstack([
    area,
    perimeter,
    mean,
    std,
    contrast,
    homogeneity
])
```

---

# 🧠 IMAGE CLASSIFICATION

## Contoh Machine Learning Pipeline

```python
X = np.random.rand(200,6)
y = np.random.randint(0,2,200)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

model = SVC(kernel='rbf')
model.fit(X_train,y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test,y_pred))
```

---

# 📊 Contoh Studi Kasus

## Klasifikasi Daun / Sel / Tekstur

Pipeline:

Image → Segmentation → Descriptor → Feature Vector → PCA → ML → Prediction

---

# 🎓 Kesimpulan

* Descriptor mengubah citra menjadi fitur numerik.
* Machine Learning bekerja pada feature vector.
* Pendekatan ini merupakan dasar Classical Computer Vision sebelum Deep Learning.

## Implementasi: Build Classifier From Scratch

---

# 🎯 Learning Outcome

Mahasiswa mampu:

* melakukan preprocessing citra
* mengekstrak berbagai descriptor
* membangun feature vector
* melakukan reduksi dimensi
* melatih model machine learning
* mengevaluasi classifier

---

# 1️⃣ Import Library

```python
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
```

---

# 2️⃣ Dataset Praktikum (AUTO GENERATED)

Mahasiswa tidak perlu download dataset.

```python
def generate_circle():
    img = np.zeros((128,128),dtype=np.uint8)
    cv2.circle(img,(64,64),30,255,-1)
    return img


def generate_square():
    img = np.zeros((128,128),dtype=np.uint8)
    cv2.rectangle(img,(34,34),(94,94),255,-1)
    return img
```

---

# 3️⃣ Membuat Dataset

```python
X_images = []
y_labels = []

for i in range(100):
    X_images.append(generate_circle())
    y_labels.append(0)

    X_images.append(generate_square())
    y_labels.append(1)
```

Visualisasi:

```python
plt.imshow(X_images[0],cmap='gray')
plt.title('Sample Image')
plt.axis('off')
```

---

# 4️⃣ REGION DESCRIPTOR

```python
def region_features(img):
    contours,_ = cv2.findContours(img,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cnt = contours[0]

    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt,True)

    x,y,w,h = cv2.boundingRect(cnt)
    aspect_ratio = w/h

    return [area, perimeter, aspect_ratio]
```

---

# 5️⃣ MOMENT DESCRIPTOR

```python
def moment_features(img):
    m = cv2.moments(img)
    hu = cv2.HuMoments(m).flatten()
    return hu
```

---

# 6️⃣ TEXTURE FEATURE (STATISTICAL)

```python
def texture_features(img):
    mean = np.mean(img)
    std = np.std(img)
    energy = np.sum(img**2)
    return [mean,std,energy]
```

---

# 7️⃣ GLCM FEATURE

```python
def glcm_features(img):
    glcm = graycomatrix(img,
                        distances=[1],
                        angles=[0],
                        levels=256,
                        symmetric=True,
                        normed=True)

    contrast = graycoprops(glcm,'contrast')[0,0]
    homogeneity = graycoprops(glcm,'homogeneity')[0,0]

    return [contrast, homogeneity]
```

---

# 8️⃣ LOCAL BINARY PATTERN (LBP)

```python
def lbp_features(img):
    lbp = local_binary_pattern(img,8,1,method='uniform')
    hist,_ = np.histogram(lbp.ravel(),bins=10,range=(0,10))
    hist = hist.astype(float)
    hist /= hist.sum()
    return hist
```

---

# 9️⃣ FEATURE VECTOR BUILDER

```python
def extract_features(img):

    f1 = region_features(img)
    f2 = moment_features(img)
    f3 = texture_features(img)
    f4 = glcm_features(img)
    f5 = lbp_features(img)

    return np.hstack([f1,f2,f3,f4,f5])

X_features = np.array([extract_features(img) for img in X_images])
y = np.array(y_labels)

print(X_features.shape)
```

---

# 🔟 PCA (Dimensionality Reduction)

```python
pca = PCA(n_components=10)
X_reduced = pca.fit_transform(X_features)
```

---

# 1️⃣1️⃣ Train-Test Split

```python
X_train,X_test,y_train,y_test = train_test_split(
    X_reduced,y,test_size=0.2,random_state=42)
```

---

# 1️⃣2️⃣ Build Classifier (SVM)

```python
model = SVC(kernel='rbf')
model.fit(X_train,y_train)
```

---

# 1️⃣3️⃣ Evaluation

```python
y_pred = model.predict(X_test)

print('Accuracy:',accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
```

---

# 1️⃣4️⃣ Visual Prediction

```python
sample = X_images[10]
feat = extract_features(sample).reshape(1,-1)
feat = pca.transform(feat)

pred = model.predict(feat)

plt.imshow(sample,cmap='gray')
plt.title(f'Prediction: {pred[0]}')
plt.axis('off')
```

---

In [1]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [2]:
DATASET_PATH = 'dataset_fruit'
CATEGORIES = ['Apple Red 1', 'Orange 1']

In [3]:
def extract_features(img_gray):
    _, binary = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) > 0:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = float(w) / h if h != 0 else 0
    else:
        area, perimeter, aspect_ratio = 0, 0, 0
    
    moments = cv2.moments(binary)
    hu_moments = cv2.HuMoments(moments).flatten()
    
    glcm = graycomatrix(img_gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    
    lbp = local_binary_pattern(img_gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    lbp_hist = lbp_hist.astype('float')
    if lbp_hist.sum() > 0:
        lbp_hist /= lbp_hist.sum()
        
    return np.hstack([area, perimeter, aspect_ratio, hu_moments, contrast, homogeneity, lbp_hist])

In [4]:
X_features = []
y_labels = []

for label_idx, category in enumerate(CATEGORIES):
    folder_path = os.path.join(DATASET_PATH, category)
    image_paths = glob.glob(os.path.join(folder_path, '*.jpg'))
    
    for img_path in image_paths:
        img_gray = cv2.imread(img_path, 0)
        if img_gray is None:
            continue
            
        feature_vec = extract_features(img_gray)
        X_features.append(feature_vec)
        y_labels.append(label_idx)

X_features = np.array(X_features)
y_labels = np.array(y_labels)

print("X shape:", X_features.shape)
print("y shape:", y_labels.shape)

X shape: (324, 22)
y shape: (324,)


In [5]:
pca = PCA(n_components=10)
X_reduced = pca.fit_transform(X_features)

X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

model = SVC(kernel='rbf', C=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=CATEGORIES))

Accuracy: 1.0
              precision    recall  f1-score   support

 Apple Red 1       1.00      1.00      1.00        33
    Orange 1       1.00      1.00      1.00        32

    accuracy                           1.00        65
   macro avg       1.00      1.00      1.00        65
weighted avg       1.00      1.00      1.00        65

